# BrainGuard: run the full study on a free Colab GPU

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.*

Then *Runtime → Run all*. Total time is roughly **1.5–2.5 hours**, mostly waiting.

- Progress is saved to your Google Drive (`MyDrive/brainguard`). **If Colab disconnects, just run all cells again**: finished steps are skipped.
- At the end, a file called `results_bundle.zip` downloads. It contains only reports and predictions (no images, no models).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: set Runtime -> Change runtime type -> T4 GPU"

## 2. Connect Google Drive (optional: keeps progress if the session disconnects)

A pop-up asks for permission: choose your Google account and tick **all** boxes. If it fails, the notebook continues without Drive (it then cannot resume after a disconnect).

In [ ]:
import os
WORKDIR = '/content/brainguard'          # fallback: local disk (lost if the session ends)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/brainguard'
    print('Google Drive connected - progress will be saved and can resume.')
except Exception as e:
    print('Google Drive not connected (', type(e).__name__, ') - continuing without it.')
    print('Keep this tab open until the run finishes.')
os.makedirs(WORKDIR, exist_ok=True)
print('Work folder:', WORKDIR)

## 3. Get the code

In [ ]:
%cd /content
!rm -rf BrainGuard-MRI-Tumor-Detection
!git clone -q https://github.com/Muneera-DSA/BrainGuard-MRI-Tumor-Detection.git
%cd /content/BrainGuard-MRI-Tumor-Detection
!pip install -q -e ".[dl]"

## 4. Provide the Kaggle dataset zip

If `Brain_tumor_Image_dataset.zip` is in your Drive folder `MyDrive/brainguard`, it is found automatically. Otherwise an **upload button** appears below: click it and choose the zip (156 MB). The figshare data is downloaded automatically.

In [ ]:
import glob, shutil
zips = glob.glob(f'{WORKDIR}/*.zip')
zips = [z for z in zips if 'results_bundle' not in z]
if zips:
    KAGGLE_ZIP = zips[0]
elif os.path.exists('/content/data/kaggle/Training'):
    KAGGLE_ZIP = ''
else:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    KAGGLE_ZIP = f'{WORKDIR}/{name}'
    shutil.move(name, KAGGLE_ZIP)
print('Kaggle zip:', KAGGLE_ZIP or '(already extracted)')

## 5. Run everything (about 1.5–2.5 hours)

In [ ]:
zip_arg = f'--kaggle-zip "{KAGGLE_ZIP}"' if KAGGLE_ZIP else ''
!python -m brainguard.pipeline {zip_arg} --workdir "{WORKDIR}" --data-dir /content/data

## 6. Download the results and send `results_bundle.zip` back

In [ ]:
import os
from google.colab import files
bundle = f'{WORKDIR}/results_bundle.zip'
if os.path.exists(bundle):
    files.download(bundle)
else:
    print('No results_bundle.zip yet: the run above did not finish.')
    print('Scroll up in the run cell and copy the lines containing "Error" or "Traceback".')

---
*Optional pipeline check (about 10 minutes) before the full run:* replace step 5 with
`!python -m brainguard.pipeline {zip_arg} --workdir "{WORKDIR}_quick" --data-dir /content/data --quick`

---
## Finish a run that stopped (or retrain the baseline CNN)

Use this cell **instead of step 5** when asked: it keeps everything already finished, retrains only the baseline CNN, completes the remaining steps (SHAP / Grad-CAM, report) and downloads the results. About 20–30 minutes.

In [ ]:
zip_arg = f'--kaggle-zip "{KAGGLE_ZIP}"' if KAGGLE_ZIP else ''
!python -m brainguard.pipeline {zip_arg} --workdir "{WORKDIR}" --data-dir /content/data --stages data train robust explain report bundle --redo cnn_patient
import os
from google.colab import files
bundle = f'{WORKDIR}/results_bundle.zip'
if os.path.exists(bundle):
    files.download(bundle)
else:
    print('No results_bundle.zip yet: the run above did not finish.')
    print('Scroll up in the run cell and copy the lines containing "Error" or "Traceback".')